In [1]:
import os
from typing import List,Dict,Any
import pandas as pd

SETUP FOR DOC INGESTION AND SPLITTING

In [1]:
# libraries for building docs and splitting text
from langchain_core.documents import Document
from langchain_text_splitters import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter
)

DOC STRUCTURE LANGCHAIN

In [3]:
# simple example doc
doc = Document(
    page_content="This is the text content which will be embedded",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Messi",
        "date_created":"2024-01-01"
    }
)
print("Doc Strucutre")
print(f"Content: {doc.page_content}")
print(f"Metadata: {doc.metadata}")

Doc Strucutre
Content: This is the text content which will be embedded
Metadata: {'source': 'example.txt', 'page': 1, 'author': 'Messi', 'date_created': '2024-01-01'}


1> Text files (.txt) Ingestion 

In [4]:
# making a directory to store text files
os.makedirs("data/text_files",exist_ok=True)

In [7]:
# generating a sample text file
sample_texts={
    "data/text_files/python_intro.txt":"""
Python is a high-level, interpreted programming language known for its simplicity, readability, and versatility. Created by Guido van Rossum and first released in 1991, Python emphasizes clear and concise syntax, making it easy for beginners to learn while remaining powerful enough for professional software development. Its extensive standard library and large ecosystem of third-party packages allow developers to build a wide variety of applications efficiently.

One of the key features of Python is its support for multiple programming paradigms, including object-oriented, procedural, and functional programming. Python offers automatic memory management, cross-platform compatibility, and a vast collection of libraries for tasks such as web development, data analysis, machine learning, artificial intelligence, automation, and scientific computing. These features significantly reduce development time and improve productivity.

Python is widely used across industries for diverse applications. It powers web applications through frameworks such as Django and Flask, supports data science and machine learning with libraries like NumPy, Pandas, and TensorFlow, and is commonly used for automation and scripting tasks. Due to its ease of use, flexibility, and strong community support, Python has become one of the most popular programming languages in the world and is extensively used in both academic and professional environments.

""",
"data/text_files/nextjs_intro.txt":"""Next.js is a powerful open-source React framework developed by Vercel that enables developers to build fast, scalable, and production-ready web applications. It extends the capabilities of React by providing features such as server-side rendering (SSR), static site generation (SSG), API routes, and file-based routing out of the box. These built-in features simplify development and help create applications that are both efficient and easy to maintain.

One of the most important aspects of Next.js is its focus on performance and search engine optimization (SEO). By allowing pages to be rendered on the server before being sent to the browser, Next.js improves initial page load times and ensures that search engines can easily index website content. Additional features such as image optimization, automatic code splitting, and incremental static regeneration further enhance application speed and user experience.

Next.js is widely used for developing modern web applications, including e-commerce platforms, dashboards, content management systems, blogs, and enterprise-level applications. Its flexibility allows developers to choose between static rendering and dynamic server-side rendering based on project requirements. Companies ranging from startups to large enterprises adopt Next.js because it combines the simplicity of React with powerful performance optimization and deployment capabilities.
"""
}
for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

1a> Loading text file (Reading data from a single file)

In [8]:
# libraries to load text file
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt",encoding="utf-8")
documents = loader.load()
print(f"Loaded: {len(documents)} document")
print(type(documents[0]))
print(f"Content: {documents[0].page_content[:100]}")
print(f"Metadata: {documents[0].metadata}")

Loaded: 1 document
<class 'langchain_core.documents.base.Document'>
Content: 
Python is a high-level, interpreted programming language known for its simplicity, readability, and
Metadata: {'source': 'data/text_files/python_intro.txt'}


C:\Users\Dell\AppData\Local\Temp\ipykernel_18944\1318614302.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


1b> Loading text files (Reading data directly from directory)

In [9]:
# libraries to load text files from directory
from langchain_community.document_loaders import DirectoryLoader

dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## pattern to match txt files
    loader_cls = TextLoader, ## Loader class to use
    loader_kwargs = {'encoding':'utf-8'},
    show_progress =  True
)

documents = dir_loader.load()
print(f"Loaded: {len(documents)} documents")
for i,doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")

100%|██████████| 2/2 [00:00<00:00, 648.82it/s]

Loaded: 2 documents

Document 1:
  Source: data\text_files\nextjs_intro.txt
  Length: 1411 characters

Document 2:
  Source: data\text_files\python_intro.txt
  Length: 1446 characters


2> Text Splitting Strategies

In [ ]:
# Libraries for text splitting already imported above

# from langchain_text_splitters import(
#     RecursiveCharacterTextSplitter,
#     CharacterTextSplitter,
#     TokenTextSplitter
# )

2a> Method 1: Character Text Splitter

In [10]:
text=documents[0].page_content
print(text)

Next.js is a powerful open-source React framework developed by Vercel that enables developers to build fast, scalable, and production-ready web applications. It extends the capabilities of React by providing features such as server-side rendering (SSR), static site generation (SSG), API routes, and file-based routing out of the box. These built-in features simplify development and help create applications that are both efficient and easy to maintain.

One of the most important aspects of Next.js is its focus on performance and search engine optimization (SEO). By allowing pages to be rendered on the server before being sent to the browser, Next.js improves initial page load times and ensures that search engines can easily index website content. Additional features such as image optimization, automatic code splitting, and incremental static regeneration further enhance application speed and user experience.

Next.js is widely used for developing modern web applications, including e-comm

In [11]:
char_splitter = CharacterTextSplitter(
    separator="\n", #splitting on new lines
    chunk_size=200, # size of chunk 200 chars
    chunk_overlap=20, # how many chars allowed to overlap
    length_function=len # how to measure chunksize
)
char_chunks=char_splitter.split_text(text)
print(f"No of chunks: {len(char_chunks)}")
print(f"First Chunk: {char_chunks[0][:100]}")
print(char_chunks[0])
print("------------")
print(char_chunks[1])

Created a chunk of size 454, which is longer than the specified 200
Created a chunk of size 463, which is longer than the specified 200


No of chunks: 3
First Chunk: Next.js is a powerful open-source React framework developed by Vercel that enables developers to bui
Next.js is a powerful open-source React framework developed by Vercel that enables developers to build fast, scalable, and production-ready web applications. It extends the capabilities of React by providing features such as server-side rendering (SSR), static site generation (SSG), API routes, and file-based routing out of the box. These built-in features simplify development and help create applications that are both efficient and easy to maintain.
------------
One of the most important aspects of Next.js is its focus on performance and search engine optimization (SEO). By allowing pages to be rendered on the server before being sent to the browser, Next.js improves initial page load times and ensures that search engines can easily index website content. Additional features such as image optimization, automatic code splitting, and incremental static regene

2b> Method 2: Recusrive Character Text Splitter


In [12]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n","\n"," ",""],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)
recursive_chunks = recursive_splitter.split_text(text)
print(f"No of Chunks: {len(recursive_chunks)}")

No of Chunks: 9


In [13]:
for i in range(0,len(recursive_chunks)):
    print(f"Chunk {i+1}: {recursive_chunks[i]}")
    print(f"Length of Chunk {i+1}: {len(recursive_chunks[i])}")
    print("---------")

Chunk 1: Next.js is a powerful open-source React framework developed by Vercel that enables developers to build fast, scalable, and production-ready web applications. It extends the capabilities of React by
Length of Chunk 1: 197
---------
Chunk 2: of React by providing features such as server-side rendering (SSR), static site generation (SSG), API routes, and file-based routing out of the box. These built-in features simplify development and
Length of Chunk 2: 197
---------
Chunk 3: development and help create applications that are both efficient and easy to maintain.
Length of Chunk 3: 86
---------
Chunk 4: One of the most important aspects of Next.js is its focus on performance and search engine optimization (SEO). By allowing pages to be rendered on the server before being sent to the browser, Next.js
Length of Chunk 4: 199
---------
Chunk 5: browser, Next.js improves initial page load times and ensures that search engines can easily index website content. Additional features such 

3> Method 3: TokenTextSplitter

In [14]:
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)
token_chunks = token_splitter.split_text(text)
print(f"No of chunks created: {len(token_chunks)}")

No of chunks created: 7
